# AgentPulse Reasoning Strategy Benchmark (Kaggle GPU) -- Llama 3.1 8B

Runs the exact same Direct / CoT / AoT comparison as the local CPU run (`experiments/reasoning_strategies.py`)
and the earlier Qwen3-8B Kaggle GPU attempt, reusing the actual project code (strategies, evaluator, risk
aggregation, and `LlamaGGUFAdapter` -- real llama.cpp inference, not a fallback) -- this run swaps in Llama 3.1
8B Instruct for cross-model generalization, on the same evaluation pipeline and dataset as the Qwen3-8B numbers
already committed.

Model: `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` (Q4_K_M). Meta doesn't publish an official GGUF release the
way the Qwen team does for Qwen3 -- bartowski is one of the most widely-used, trusted third-party quantizers in
the llama.cpp community (a correct quantization of the official Meta weights, not a fine-tune or unofficial
variant), so this is the closest equivalent available.

llama.cpp's CUDA backend was chosen over `transformers` + `bitsandbytes` because it supports a much wider range
of NVIDIA GPU generations. Kaggle assigns whichever GPU is available (P100, T4, etc.) and bitsandbytes' compiled
kernels don't support older ones (Pascal/P100, compute capability sm_60) -- this was tried first on the Qwen3
attempt and crashed with `Error named symbol not found` for exactly that reason.

**Root-cause fix from the discarded Qwen3-8B GPU run**: that run completed without erroring but silently
produced 450/450 fake `0.0` grounding-risk scores, because `grounding.py`'s NLI/embedding models failed to load
on Kaggle (fail-open by design -- a deliberate production safety property, but wrong for a benchmark that
should fail loudly instead) and the benchmark loop's `eval_res.overall_risk_score or 0.0` masked that failure as
an indistinguishable real zero. This notebook now asserts `grounding.models_loaded()` before the benchmark loop
and raises immediately if either model isn't loaded, instead of silently proceeding -- see the cell below the
model-loading cell.

Requires the `agentpulse-code` dataset attached (Add Input, re-uploaded with the current code including
`LlamaGGUFAdapter`) and a GPU accelerator enabled (Settings > Accelerator) before Run All.


In [ ]:
# Build llama-cpp-python with CUDA support. This compiles from source and can take
# a few minutes; a prebuilt CPU-only wheel would silently ignore n_gpu_layers.
# Fails loudly (rather than continuing past a broken build) if the wheel doesn't
# build or llama_cpp can't be imported afterward -- an earlier attempt without
# this check silently kept running for several more minutes downloading a 5GB
# model before finally crashing on the import.
import os, subprocess, sys

# CMAKE_CUDA_ARCHITECTURES must be set explicitly: llama.cpp's default target
# list can miss older cards (Kaggle may assign a P100, compute capability 6.0),
# and the build fails outright rather than skipping unsupported architectures.
# 60=P100 (Pascal), 75=T4 (Turing) covers Kaggle's two common GPU assignments.
import torch

# Building from source failed with "CMake Error ... CUDA::cuda_driver ... target
# was not found" -- CMake's CUDAToolkit package couldn't locate the driver stub
# library in this containerized build environment. Rather than fight the build,
# use a prebuilt CUDA wheel (avoids compilation entirely).
cuda_tag = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cu124"
wheel_index = f"https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}"
print(f"Detected CUDA {torch.version.cuda}, trying prebuilt wheel index: {wheel_index}")

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-v", "--force-reinstall", "--no-cache-dir",
     "llama-cpp-python", "--extra-index-url", wheel_index],
    capture_output=True, text=True,
)
print("=== TAIL OF OUTPUT ===")
print(result.stdout[-6000:])
print(result.stderr[-2000:])

if result.returncode != 0:
    print("\nPrebuilt wheel install failed too -- falling back to source build with explicit CUDA stub path.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=True)
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=60;75"
    # The stub libcuda.so (link-time only; the real driver is loaded at runtime)
    # usually exists under the CUDA toolkit's lib64/stubs, but isn't always on
    # CMake's search path by default -- point it there explicitly.
    for stub_dir in ["/usr/local/cuda/lib64/stubs", "/usr/local/cuda-12.4/lib64/stubs", "/usr/local/cuda-12.1/lib64/stubs"]:
        if os.path.isdir(stub_dir):
            os.environ["LIBRARY_PATH"] = stub_dir + os.pathsep + os.environ.get("LIBRARY_PATH", "")
            os.environ["CMAKE_ARGS"] += f" -DCMAKE_LIBRARY_PATH={stub_dir}"
            print("Using CUDA stub dir:", stub_dir)
            break
    result2 = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-v", "--force-reinstall", "--no-cache-dir", "llama-cpp-python"],
        capture_output=True, text=True,
    )
    print("=== FALLBACK BUILD TAIL ===")
    print(result2.stdout[-6000:])
    print(result2.stderr[-2000:])
    if result2.returncode != 0:
        raise RuntimeError("llama-cpp-python install failed via both prebuilt wheel and source build -- see output above.")

import llama_cpp  # fail here, not several minutes later after downloading the model
print("llama_cpp imported OK:", llama_cpp.__file__)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)


In [ ]:
import os, glob, zipfile

# The dataset may be a set of per-folder zips (backend.zip, sdk.zip, ...) or already
# extracted, depending on how it was uploaded. Handle both: extract any zips found
# under /kaggle/input into /kaggle/working/agentpulse_src first, then search there too.
extract_dir = "/kaggle/working/agentpulse_src"
os.makedirs(extract_dir, exist_ok=True)

for zpath in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    name = os.path.splitext(os.path.basename(zpath))[0]
    dest = os.path.join(extract_dir, name)
    if not os.path.isdir(dest):
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(dest)
        print(f"Extracted {zpath} -> {dest}")

search_roots = ["/kaggle/input", extract_dir]
project_root = None
for base in search_roots:
    for backend_dir in glob.glob(f"{base}/**/backend", recursive=True):
        root = os.path.dirname(backend_dir)
        if os.path.isdir(os.path.join(root, "sdk")) and os.path.isdir(os.path.join(root, "reasoning")):
            project_root = root
            break
    if project_root:
        break

if not project_root and all(
    os.path.isdir(os.path.join(extract_dir, d)) for d in ("backend", "sdk", "reasoning", "datasets", "llm_adapters")
):
    project_root = extract_dir

assert project_root, (
    "Could not find the project. Make sure you added the 'agentpulse-code' dataset "
    "(Add Input) -- it should contain backend/, sdk/, reasoning/, datasets/, llm_adapters/ "
    "(as folders or as backend.zip/sdk.zip/reasoning.zip/datasets.zip/llm_adapters.zip)."
)
print("Project root:", project_root)
print("Contents:", os.listdir(project_root))


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_root}/sdk"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_root}/backend"], check=True)

sys.path.insert(0, project_root)
sys.path.insert(0, f"{project_root}/backend")
sys.path.insert(0, f"{project_root}/sdk/src")
print("Installed and added to path.")


In [ ]:
from huggingface_hub import hf_hub_download

# bartowski/Meta-Llama-3.1-8B-Instruct-GGUF -- confirmed exact filename via the
# HF Hub API before this run (Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf, ~4.92GB)
# rather than guessing, since a typo here would waste hours of GPU quota on a
# download 404 partway through Run All.
model_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    local_dir="/kaggle/working/models",
)
print("Model downloaded to:", model_path)


In [ ]:
import json, time, statistics

from llm_adapters.llama import LlamaGGUFAdapter
from reasoning import get_reasoning_strategy
from app.services.evaluator import EvaluationPipeline
from app.services.drift import DriftDetector
from app.services.alerting import AlertEngine
from app.services import grounding

# n_gpu_layers=-1 offloads every layer to GPU. Same adapter family (real
# llama.cpp inference, not LocalHFAdapter's fallback path), same chat
# template handling as the local Qwen3 run -- only the model, n_gpu_layers,
# and n_threads differ from a CPU configuration.
adapter = LlamaGGUFAdapter(
    model_path=model_path,
    device="cuda",
    quantization="Q4_K_M",
    n_gpu_layers=-1,
    load_immediately=True,
)
print(f"Loaded on GPU in {adapter._load_time_ms:.0f}ms")


In [ ]:
grounding.load_models(use_onnx=False, sync=True)

# Root-cause fix for the discarded Qwen3-8B run: that run's evaluation models
# failed to load on Kaggle (execution log came back empty, exact cause never
# confirmed), and because grounding.py is deliberately fail-open (a real
# production-safety property -- an evaluation failure should never block the
# pipeline), evaluate_grounding() silently returned None for all 450 calls.
# The benchmark loop's `eval_res.overall_risk_score or 0.0` then turned every
# one of those silent failures into a value indistinguishable from a genuine
# zero-risk result. Assert loaded state explicitly instead of discovering this
# after burning the GPU quota on a full run.
status = grounding.models_loaded()
print("Model load status:", status)
assert status["nli_model"] and status["nli_tokenizer"], (
    f"NLI model failed to load on Kaggle -- models_loaded() returned {status}. "
    "Aborting rather than producing a benchmark full of fail-open 0.0 risk scores."
)
assert status["embedding_model"], (
    f"Embedding model failed to load on Kaggle -- models_loaded() returned {status}. "
    "Aborting rather than producing a benchmark full of fail-open 0.0 risk scores."
)
print("Evaluation models confirmed loaded -- safe to proceed.")

with open(f"{project_root}/datasets/v1.0_test.json") as f:
    dataset = json.load(f)
cases = dataset["cases"]
print(f"{len(cases)} test cases loaded")

drift_detector = DriftDetector(window_size=20, min_samples_for_alert=5)
alert_engine = AlertEngine(cooldown_seconds=0)
pipeline = EvaluationPipeline(drift_detector, alert_engine)

warm = adapter.generate_with_metadata(prompt="Reply with the single word: ready.", max_tokens=8)
print(f"Warm-up: {warm.latency_ms:.0f}ms, {warm.tokens_out} tokens -> {warm.raw_metadata['tokens_per_sec']} tok/s")

# Sanity-check the warm-up call actually produced a real grounding score (not
# None), on top of the models_loaded() check above -- belt and suspenders,
# since models_loaded() only proves the models are in memory, not that
# inference through the full pipeline actually works end to end.
warm_eval = pipeline.evaluate_span(
    span_id="warmup_check", trace_id="warmup_check", agent_id="warmup",
    input_text="The system is ready.", output_text=warm.text,
)
assert warm_eval.grounding is not None and warm_eval.grounding.grounding_score is not None, (
    "Warm-up evaluation produced no grounding score even though models_loaded() "
    "reported everything loaded -- something else is wrong with the pipeline. Aborting."
)
print(f"Warm-up grounding score: {warm_eval.grounding.grounding_score} (stage={warm_eval.grounding.evaluation_stage}) -- pipeline confirmed working end to end."
)


## Benchmark loop

Same structure as `experiments/reasoning_strategies.py` and the local CPU run -- identical strategy calls,
identical evaluator, identical per-case/per-run statistics. `N_RUNS` and `MAX_TOKENS` match both prior runs so
all three (local CPU Qwen3-8B, Kaggle GPU Qwen3-8B, Kaggle GPU Llama 3.1 8B) are directly comparable.


In [ ]:
N_RUNS = 5
MAX_TOKENS = 200
strategies = ["direct", "cot", "aot"]
results_by_strategy = {s: [] for s in strategies}

def _stdev(xs):
    return round(float(statistics.stdev(xs)), 2) if len(xs) > 1 else 0.0

overall_start = time.perf_counter()

for strat_name in strategies:
    strat = get_reasoning_strategy(strat_name)
    print(f"\n=== {strat_name.upper()} ===")

    for case in cases:
        run_latencies, run_tin, run_tout, run_risks, run_contra = [], [], [], [], []

        for run_idx in range(N_RUNS):
            output = strat.execute(
                adapter=adapter,
                task_prompt=case["input_query"],
                context=case.get("evidence"),
                max_tokens=MAX_TOKENS,
            )
            eval_res = pipeline.evaluate_span(
                span_id=f"{strat_name}_{case['id']}_{run_idx}",
                trace_id=f"trace_{strat_name}_{run_idx}",
                agent_id="eval_agent",
                input_text=case.get("evidence") or case["input_query"],
                output_text=output.final_answer,
                tool_calls=case.get("tool_records"),
            )
            risk = eval_res.overall_risk_score
            assert risk is not None, (
                f"overall_risk_score was None for {strat_name}/{case['id']}/run{run_idx} -- "
                "the evaluation pipeline stopped producing scores mid-run. Aborting rather "
                "than silently writing a fake 0.0 the way the discarded Qwen3 run did."
            )
            run_latencies.append(output.latency_ms)
            run_tin.append(output.tokens_in)
            run_tout.append(output.tokens_out)
            run_risks.append(risk)
            contra_p = eval_res.grounding.contradiction_prob if eval_res.grounding else 0.0
            run_contra.append(1.0 if (contra_p or 0) > 0.60 else 0.0)

        print(f"  {case['id']}: lat={statistics.mean(run_latencies):.0f}ms  risk={statistics.mean(run_risks):.3f}")

        results_by_strategy[strat_name].append({
            "case_id": case["id"],
            "domain": case["domain"],
            "is_failure_ground_truth": case["is_failure"],
            "n_runs": N_RUNS,
            "avg_latency_ms": round(statistics.mean(run_latencies), 2),
            "median_latency_ms": round(statistics.median(run_latencies), 2),
            "stdev_latency_ms": _stdev(run_latencies),
            "avg_tokens_in": round(statistics.mean(run_tin), 1),
            "avg_tokens_out": round(statistics.mean(run_tout), 1),
            "stdev_tokens_out": _stdev(run_tout),
            "avg_risk_score": round(statistics.mean(run_risks), 3),
            "stdev_risk_score": round(_stdev(run_risks), 3),
            "contradiction_rate": round(statistics.mean(run_contra), 3),
            "raw_latencies_ms": [round(x, 2) for x in run_latencies],
            "raw_risk_scores": [round(x, 3) for x in run_risks],
        })

total_wall_s = time.perf_counter() - overall_start
print(f"\nTotal wall time: {total_wall_s/60:.1f} minutes")


In [ ]:
summary = {}
for s_name, case_results in results_by_strategy.items():
    all_lats = [c["avg_latency_ms"] for c in case_results]
    all_tin = [c["avg_tokens_in"] for c in case_results]
    all_tout = [c["avg_tokens_out"] for c in case_results]
    all_risks = [c["avg_risk_score"] for c in case_results]
    all_contras = [c["contradiction_rate"] for c in case_results]

    summary[s_name.upper()] = {
        "mean_latency_ms": round(statistics.mean(all_lats), 2),
        "median_latency_ms": round(statistics.median(all_lats), 2),
        "stdev_latency_ms": round(float(statistics.stdev(all_lats)), 2) if len(all_lats) > 1 else 0.0,
        "mean_tokens_in": round(statistics.mean(all_tin), 1),
        "mean_tokens_out": round(statistics.mean(all_tout), 1),
        "mean_grounding_risk": round(statistics.mean(all_risks), 3),
        "stdev_grounding_risk": round(float(statistics.stdev(all_risks)), 3) if len(all_risks) > 1 else 0.0,
        "contradiction_rate": round(statistics.mean(all_contras), 3),
    }

import torch
out_payload = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "model": "llama3.1-8b-gpu",
    "model_id": adapter.model_id,
    "adapter": type(adapter).__name__,
    "provider": "kaggle_gpu_llamacpp",
    "real_inference": True,
    "warmup_ms": round(warm.latency_ms, 2),
    "hardware": {
        "platform": "Kaggle",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "quantization": "Q4_K_M (llama.cpp, full GPU offload)",
    },
    "evaluation_models_confirmed_loaded": status,
    "dataset": "v1.0_test",
    "n_cases": len(cases),
    "runs_per_case": N_RUNS,
    "max_tokens_per_call": MAX_TOKENS,
    "total_wall_time_minutes": round(total_wall_s / 60, 1),
    "summary": summary,
    "detailed_results": results_by_strategy,
}

out_path = "/kaggle/working/reasoning_strategy_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(out_payload, f, indent=2)

print(f"Saved to {out_path} -- download it from the Output panel.")
print(json.dumps(summary, indent=2))


In [ ]:
# Delete the 5GB model file so it isn't included as kernel output -- otherwise
# `kaggle kernels output` has to transfer it just to fetch the small results JSON.
import shutil
shutil.rmtree("/kaggle/working/models", ignore_errors=True)
print("Cleaned up model weights from output.")
